# CSV and Excel Parsing: Choose the Right Row Grain

| Field | Value |
|---|---|
| Stage | Data foundation |
| Difficulty | Beginner to intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
A table is not one blob. Preserve schema and row identity so filtering, retrieval, and citations operate at the intended grain.

## 30-Second Summary

This notebook loads equivalent product data from CSV and Excel, validates schema and value parity, then builds one retrieval document per row with typed metadata.

## Why This Matters

Flattening a spreadsheet hides columns, nulls, and row identity. Row-aware documents support exact filters and explain which record supplied an answer.

## Scope

| Covers | Does not cover |
|---|---|
| Schema checks, null checks, CSV/Excel parity, row documents | Formulas, merged cells, multi-sheet business logic, live spreadsheet sessions |


## Mental Model

```text
CSV/XLSX -> dataframe -> validate schema/types/nulls -> row records -> text + filter metadata
```


In [1]:
from pathlib import Path
import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file(): return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "05-DataIngestParsing/data/structured_files"
CSV_PATH, XLSX_PATH = DATA_DIR / "products.csv", DATA_DIR / "inventory.xlsx"
csv_frame = pd.read_csv(CSV_PATH)
xlsx_frame = pd.read_excel(XLSX_PATH)
csv_frame


,Product,Category,Price,Stock,Description
0,Laptop,Electronics,999.99,50,High-performance laptop with 16GB RAM and 512G...
1,Mouse,Accessories,29.99,200,Wireless optical mouse with ergonomic design
2,Keyboard,Accessories,79.99,150,Mechanical keyboard with RGB backlighting
3,Monitor,Electronics,299.99,75,27-inch 4K monitor with HDR support
4,Webcam,Electronics,89.99,100,1080p webcam with noise cancellation


## How It Works

We declare the expected columns, load each format with an explicit engine path, validate shape/nulls/types, and only then serialize rows. Numeric values remain typed metadata even though the retrieval text is human-readable.


## Baseline

The baseline serializes the entire table as one document. It is easy to preview but gives a retriever one oversized result and no row-level citation.


In [2]:
baseline_documents = [
    {"source": path.relative_to(REPO_ROOT).as_posix(), "content": frame.to_string(index=False)}
    for path, frame in ((CSV_PATH, csv_frame), (XLSX_PATH, xlsx_frame))
]
[(item["source"], len(item["content"])) for item in baseline_documents]


[('05-DataIngestParsing/data/structured_files/products.csv', 521),
 ('05-DataIngestParsing/data/structured_files/inventory.xlsx', 521)]

## Technique Implementation

Each row becomes a document keyed by source and row number. Product/category/price/stock stay available for filters, while labeled text provides a lexical or embedding representation.


In [3]:
EXPECTED_COLUMNS = ["Product", "Category", "Price", "Stock", "Description"]

def row_documents(path: Path, frame: pd.DataFrame) -> list[dict]:
    source = path.relative_to(REPO_ROOT).as_posix()
    docs = []
    for row_number, row in frame.iterrows():
        record = row.to_dict()
        docs.append({
            "id": f"{path.stem}:row:{row_number + 2}", "source": source,
            "row_number": row_number + 2, "metadata": record,
            "content": " | ".join(f"{column}: {record[column]}" for column in EXPECTED_COLUMNS),
        })
    return docs

csv_documents = row_documents(CSV_PATH, csv_frame)
xlsx_documents = row_documents(XLSX_PATH, xlsx_frame)
[(doc["id"], doc["content"]) for doc in csv_documents]


[('products:row:2',
  'Product: Laptop | Category: Electronics | Price: 999.99 | Stock: 50 | Description: High-performance laptop with 16GB RAM and 512GB SSD'),
 ('products:row:3',
  'Product: Mouse | Category: Accessories | Price: 29.99 | Stock: 200 | Description: Wireless optical mouse with ergonomic design'),
 ('products:row:4',
  'Product: Keyboard | Category: Accessories | Price: 79.99 | Stock: 150 | Description: Mechanical keyboard with RGB backlighting'),
 ('products:row:5',
  'Product: Monitor | Category: Electronics | Price: 299.99 | Stock: 75 | Description: 27-inch 4K monitor with HDR support'),
 ('products:row:6',
  'Product: Webcam | Category: Electronics | Price: 89.99 | Stock: 100 | Description: 1080p webcam with noise cancellation')]

## Controlled Experiment

The CSV and Excel files are intended to represent the same five records. We verify schema, nulls, and value parity before comparing whole-table context with the focused row returned for an RGB mechanical keyboard query.


In [4]:
pd.testing.assert_frame_equal(csv_frame, xlsx_frame, check_dtype=False)
query_terms = {"mechanical", "keyboard", "rgb"}
top_row = max(csv_documents, key=lambda doc: len(query_terms & set(doc["content"].lower().replace("|", " ").split())))
experiment_result = {
    "rows": len(csv_frame),
    "columns": len(csv_frame.columns),
    "null_cells": int(csv_frame.isna().sum().sum()),
    "top_product": top_row["metadata"]["Product"],
    "row_characters": len(top_row["content"]),
    "table_characters": len(baseline_documents[0]["content"]),
}
experiment_result


{'rows': 5,
 'columns': 5,
 'null_cells': 0,
 'top_product': 'Keyboard',
 'row_characters': 126,
 'table_characters': 521}

## Evaluation

Both formats contain the same **5 × 5** values with no null cells. The focused query retrieves the `Keyboard` row, whose context is smaller and carries exact price/stock metadata. Real workbooks need sheet, formula, date, and merged-cell policies.


In [5]:
assert list(csv_frame.columns) == EXPECTED_COLUMNS
assert experiment_result["rows"] == 5 and experiment_result["columns"] == 5
assert experiment_result["null_cells"] == 0
assert experiment_result["top_product"] == "Keyboard"
assert experiment_result["row_characters"] < experiment_result["table_characters"]
print("Structured-file checks passed for CSV and Excel.")


Structured-file checks passed for CSV and Excel.


## Decision Guide

| Question grain | Document grain |
|---|---|
| Product lookup | One row |
| Category summary | Grouped aggregate plus source rows |
| Narrative notes sheet | Section/paragraph |
| Exact numeric filtering | Typed database/dataframe filter before semantic ranking |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Prices become text | Type inference/configuration | Validate dtypes and units |
| Rows cite wrong source | Row IDs omitted | Store source, sheet, and row |
| Formulas look stale | Cached values used | Define recalculation policy |
| Null becomes `nan` text | Serialization before validation | Handle missingness explicitly |


## Production Notes

### Observability
Record file/sheet, row/column counts, schema drift, nulls, duplicates, and coercion failures.

### Safety and Guardrails
Formula cells can contain external links; never execute spreadsheet formulas during ingestion.

### Latency and Cost
Filter and aggregate structured fields before embedding; do not pay to encode values that exact predicates answer better.


## Practice

Add a duplicate product and a null price. Define whether to reject, deduplicate, or retain each row before changing code.

## Recall

Toggle - Recall: Why keep numbers typed?
Exact filters and comparisons are safer than asking embeddings to represent arithmetic.

Toggle - Recall: What determines document grain?
The questions and lifecycle operations the system must support.

## Sources

- [pandas I/O documentation](https://pandas.pydata.org/docs/user_guide/io.html)
- Repository fixtures: `products.csv`, `inventory.xlsx`

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the single-sheet fixtures | Add formula, date, null, and multi-sheet cases |
